# Data + Logistic Regression

Owner: Madushanka D.M.C  (proposal sections 4.2 to 4.5, 4.8, 4.9)

Put `PhiUSIIL_Phishing_URL_Dataset.csv` in `data/raw/` before running. Run from `notebooks/`.

Target convention: `data.get_target()` gives **1 = phishing**, 0 = legitimate
(raw `label` 0 = phishing). Precision, recall and ROC-AUC all refer to phishing.

## To do

- [x] Inspect the dataset: columns, data types, target distribution (4.2)
- [x] Check the available files and combine only compatible ones (4.3): one file only
- [x] Find and handle duplicates, missing and inconsistent values (4.4)
- [ ] Group decision: near-duplicate URLs (case / trailing slash), see section 4
- [ ] Group decision: keep or drop `URLSimilarityIndex`, see section 7
- [x] Encoding / scaling decisions, consistent train/test split (4.5, 4.8)
- [x] Move the final cleaning code into `src/data.py`
- [x] Tune Logistic Regression with 5-fold CV on the training split (`src/models/logreg.py`)
- [ ] Final test-set report (after the group decisions above)

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src import config, data, evaluate
from src.models import logreg

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# Quick checks only (section 5). Inspection uses the full raw file and tuning
# uses the full training split (group decision), whatever SAMPLE is.
SAMPLE = 20000

# Keep False until tuning and the group decisions are final. The test set is
# for the final report only, never for choosing hyperparameters or features.
RUN_FINAL_TEST = False

## 2. Dataset inspection (4.2, 4.3)

There is a single CSV (`config.RAW_FILE`), so there are no files to merge (4.3).

In [ ]:
raw = data.load_raw()
raw_snapshot = raw.copy(deep=True)   # used below to prove clean() does not modify its input
print("shape:", raw.shape)
print("dtypes:", raw.dtypes.astype(str).value_counts().to_dict())
print("missing values (all columns):", int(raw.isna().sum().sum()))
print("missing target:", int(raw[config.TARGET].isna().sum()))
print("raw label counts (1 = legitimate, 0 = phishing):", raw[config.TARGET].value_counts().to_dict())
print("fully identical rows:", int(raw.duplicated().sum()))
print("rows whose URL already appeared earlier:", int(raw["URL"].duplicated().sum()))

## 3. Exact duplicate URLs (4.4)

`clean()` keeps the **first occurrence** of each repeated URL (file order), so
the same URL can never be in both train and test. The table below checks what
that rule throws away.

In [ ]:
dups = data.duplicate_url_report(raw)
print("repeated URLs:", len(dups))
print("group sizes:", dups["n_rows"].value_counts().to_dict())
print("URLs whose copies have different labels:", int((dups["n_labels"] > 1).sum()))
print("URLs whose copies have different feature values:", int((dups["n_features_differing"] > 0).sum()))

# How often each feature differs between the copies of the same URL.
dups["features_differing"].explode().value_counts().head(15)

In [ ]:
# One example: the URL string is identical, but URLLength and other features are not.
example_url = dups.loc[dups["n_features_differing"].idxmax(), "URL"]
cols = ["URL", config.TARGET] + dups.loc[dups["URL"] == example_url, "features_differing"].iloc[0][:8]
raw.loc[raw["URL"] == example_url, cols]

**Decision (approved): keep the first occurrence; record that the copies differ.**

- *What:* for each repeated URL, the first row in file order is kept and the later row is dropped.
- *Why:* the copies never disagree on the label, so no label information is lost.
  Keeping one copy prevents the same URL from being in both train and test,
  which would inflate the scores. "Keep first" is deterministic, so every member gets the same rows.
- *Caveat:* the copies are **not identical records**. Their feature values differ
  (see the counts above; even `URLLength` differs for the same URL string), so the
  dataset's features were probably computed in separate crawls or by different
  versions of the feature extractor. The kept row is a convention, not the "correct" one.
- *Alternatives:* averaging the copies would invent feature values that were never
  observed. Dropping both copies loses a correctly labelled URL. Keeping both copies
  risks them being split across train and test. "Keep first" is the simplest rule that avoids all three.

In [ ]:
df = data.clean(raw)
pd.testing.assert_frame_equal(raw, raw_snapshot)   # clean() returned a copy; raw is unchanged
data.check_target(df)                              # labels are only 0 and 1 after cleaning
print("rows: raw", len(raw), "-> clean", len(df), f"(removed {len(raw) - len(df)})")
print("target after cleaning (1 = phishing):", data.get_target(df).value_counts().to_dict())

*Findings, full data, run of 2026-11-17 (outputs are cleared on save; rerun to regenerate):*
the file has 235,795 rows, no missing values and no fully identical rows. 425 URLs appear exactly
twice; no pair disagrees on the label, but **all 425 pairs differ in at least one feature value**
(`URLLength` in every pair). All 425 removed rows are phishing, so cleaning leaves 235,370 rows
(100,520 phishing, 134,850 legitimate). Labels are only 0/1 after cleaning (`check_target` passes).

## 4. Near-duplicate URLs (report only, `clean()` unchanged)

A near-duplicate pair is two **different** URL strings that become equal after
lower-casing and removing trailing `/`. They are kept in the data for now; the
group decides, because changing `clean()` changes every member's dataset and results.

In [ ]:
pairs = data.near_duplicate_url_pairs(df)
summary = {
    "near-duplicate pairs": len(pairs),
    "pairs with conflicting labels": int(pairs["labels_conflict"].sum()),
    "differ by letter case": int(pairs["differs_by_case"].sum()),
    "differ by trailing slash(es)": int(pairs["differs_by_trailing_slash"].sum()),
    "root path only (e.g. https://a.com vs https://a.com/)": int(pairs["root_path_only"].sum()),
    "with a non-empty path, query or fragment": int((~pairs["root_path_only"]).sum()),
}
pd.Series(summary, name="count")

In [ ]:
# The pair(s) with conflicting labels (raw label: 1 = legitimate, 0 = phishing).
conflict = pairs[pairs["labels_conflict"]]
conflict_rows = pd.concat([conflict["row_a"], conflict["row_b"]])
df.loc[conflict_rows, ["URL", "Domain", config.TARGET, "URLSimilarityIndex", "URLLength"]]

In [ ]:
# Would the shared split put the two halves of a pair on different sides?
# Only row indices are used here; no test labels or scores are looked at.
X_tr_idx, X_te_idx, _, _ = data.split(df, data.get_original_features(df))
in_test = df.index.isin(X_te_idx.index)
crosses = in_test[pairs["row_a"]] != in_test[pairs["row_b"]]
print("pairs split across train and test:", int(crosses.sum()), "of", len(pairs))
pairs.loc[~pairs["root_path_only"], ["url_a", "url_b", "label_a", "label_b"]].head(8)

*Findings, run of 2026-11-17:* 1,165 near-duplicate pairs. **None differ by letter case; all
differ only in trailing slashes** (one pair, `https://grin-6cbd8.web.app/#////` vs `.../#///`, differs in
the number of slashes inside a fragment). 461 pairs are root-only (`https://a.com` vs `https://a.com/`),
704 have a path, query or fragment. **One pair has conflicting labels:** `https://www.education.gouv.fr/`
is labelled phishing and `https://www.education.gouv.fr` legitimate; both have `URLSimilarityIndex` = 100,
which in the training split is true of every legitimate URL, so the phishing label looks like the error
(not verified). The shared split puts the two halves of 392 of the 1,165 pairs on different sides of train/test.

**Why case changes and trailing slashes are not automatically "the same URL"**

- *Case:* in a URL, the scheme and host are case-insensitive (`HTTP://Example.COM` = `http://example.com`),
  but the **path and query are case-sensitive** (RFC 3986). `/Login` and `/login` can be different pages on
  a case-sensitive web server (most Linux servers). Lower-casing the whole string would merge them.
  (The case count above shows whether this actually happens in PhiUSIIL.)
- *Trailing slash:* `https://a.com/path` and `https://a.com/path/` are different URIs. A server may
  redirect one to the other, serve different content, or return 404 for one of them. Only for an empty
  path (`https://a.com` vs `https://a.com/`) does RFC 3986 normalisation treat them as equivalent.
  Slashes after `#` are a fragment and never reach the server at all.
- *Model inputs differ:* many PhiUSIIL features are computed from the URL string (`URLLength`,
  character counts and ratios, `URLCharProb`, ...), so the two rows are different inputs, not copies.
  Merging needs a rule for which feature values to keep.
- *Phishing behaviour:* attackers deliberately reuse a URL with small variations, so variants may be
  genuinely separate submissions.
- *Label conflict:* the conflicting pair above is the same root URL labelled both ways. One of the two
  labels must be wrong; a blind normalise-and-drop rule would keep whichever came first.

**Counter-argument (leakage):** near-identical URLs on both sides of the split let a model partly
memorise instead of generalise, which can inflate test scores (see the "split across train and test" count).

**Options for the group:** (a) keep as is (current); (b) normalise only the safe cases
(host case, empty path `x` vs `x/`) and deduplicate; (c) keep the rows but split by normalised URL
(group-aware split) so variants stay on the same side; (d) drop the conflicting pair only.
Each option changes `clean()` or `split()`, so it must be agreed and then applied to everyone.

## 5. Quick baseline check on a sample

A fast smoke test on `SAMPLE` rows with the old untuned baseline (lbfgs, L2, C = 1).
Not used for any decision.

In [ ]:
def baseline_build():
    """The untuned starting point (previous logreg.build()), kept for comparison."""
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=config.RANDOM_STATE),
    )

df_small = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True) if SAMPLE and SAMPLE < len(df) else df
Xs_train, _, ys_train, _ = data.split(df_small, data.get_original_features(df_small))
pd.Series(evaluate.cv_evaluate(baseline_build(), Xs_train, ys_train)).round(4)

## 6. Hyperparameter tuning (4.9), full training split

- **Data:** the full training split of `data.split()` (80 %, stratified, `RANDOM_STATE`).
  The test split is created but never used here.
- **CV:** `evaluate.cv_splitter()` (5 stratified, shuffled, seeded folds), shared by all members.
- **Grid:** `C` (inverse regularisation strength, small = simpler model) x penalty (L2 shrinks all
  weights; L1 can set weights to exactly zero, i.e. built-in feature selection). Solver `liblinear` supports both.
  L2 uses C = 0.001 to 100; L1 stops at C = 1 because a single L1 fit with C = 10 did not finish in
  30 minutes on this almost separable data (see `logreg.PARAM_GRID`).
- **Selection metric:** F1 on the phishing class (balances missed phishing against false alarms).
  All five CV metrics are shown for every candidate. On an exact tie GridSearchCV keeps the first
  candidate in grid order, i.e. the smaller C (stronger regularisation).
- **Leakage:** the scaler sits inside the pipeline, so in every fold it is fitted on that fold's training part only.

10 candidates x 5 folds = 50 fits plus one refit on the full training split (a few minutes).

**Warning:** if every candidate scores about 0.999, the ranking is not meaningful (differences are
within one standard deviation) and near-perfect scores call for a leakage check: see section 7.

In [ ]:
original = data.get_original_features(df)
X_train, X_test, y_train, y_test = data.split(df, original)
print("features:", len(original), "| train rows:", len(X_train), "| test rows (untouched):", len(X_test))

search = logreg.tune(X_train, y_train)

cv_table = pd.DataFrame(search.cv_results_)
show = (
    ["param_logisticregression__C", "param_logisticregression__l1_ratio", "rank_test_f1"]
    + [f"mean_test_{m}" for m in evaluate.CV_METRICS]
    + ["std_test_f1", "mean_fit_time"]
)
cv_table[show].sort_values("rank_test_f1").round(5)

In [ ]:
# build() must contain the values the search picked (fixed values keep main.py fast).
print("best params:", search.best_params_, "| best mean CV F1:", round(search.best_score_, 5))
assert search.best_params_ == {
    "logisticregression__C": logreg.BEST_C,
    "logisticregression__l1_ratio": logreg.BEST_L1_RATIO,
}, "Update BEST_C / BEST_L1_RATIO in src/models/logreg.py"

In [ ]:
# Tuned build() vs the untuned baseline, same folds, training split only.
cv_compare = pd.DataFrame({
    "baseline (lbfgs, L2, C=1)": evaluate.cv_evaluate(baseline_build(), X_train, y_train),
    "tuned logreg.build()": evaluate.cv_evaluate(logreg.build(), X_train, y_train),
})
cv_compare.round(5)

*Findings, run of 2026-11-17 (50 original features, 188,296 training rows):* best = L2, C = 10,
mean CV F1 0.99991 (tied with C = 100). The untuned baseline reaches 0.99986, a gain of 0.00005,
about one standard deviation, so **tuning makes no practical difference**. C = 10 / L2 is kept
because it is the documented search result, not because it is clearly better. Every candidate
scores at least 0.998 on all metrics: a leakage warning, investigated in section 7.

## 7. `URLSimilarityIndex`: evidence for the group (training split only)

`URLSimilarityIndex` stays in the shared feature list. This section only collects
evidence; the group decides. Checks:
1. single-feature ROC-AUC of every original feature on the training split
   (direction-free: `max(AUC, 1 - AUC)`), to see whether one feature almost reveals the label;
2. how `URLSimilarityIndex` is distributed per class;
3. the tuned Logistic Regression with and without it (same 5 CV folds).

In [ ]:
single_auc = {
    col: max(auc, 1 - auc)
    for col in original
    for auc in [roc_auc_score(y_train, X_train[col])]
}
pd.Series(single_auc, name="single-feature ROC-AUC (train)").sort_values(ascending=False).head(10).round(4)

In [ ]:
usi = X_train["URLSimilarityIndex"]
cls = y_train.map({1: "phishing", 0: "legitimate"})
print("share of rows with URLSimilarityIndex == 100, by class:")
print((usi == 100).groupby(cls).mean().round(4).to_dict())
usi.groupby(cls).describe().round(2)

In [ ]:
without = [c for c in original if c != "URLSimilarityIndex"]
usi_compare = pd.DataFrame({
    f"with URLSimilarityIndex ({len(original)} features)": evaluate.cv_evaluate(logreg.build(), X_train[original], y_train),
    f"without URLSimilarityIndex ({len(without)} features)": evaluate.cv_evaluate(logreg.build(), X_train[without], y_train),
})
usi_compare["difference (without - with)"] = usi_compare.iloc[:, 1] - usi_compare.iloc[:, 0]
usi_compare.round(5)

**How to read this for the group discussion**

- A single-feature ROC-AUC close to 1.0 means the feature alone nearly separates the classes.
  That is a warning sign of a feature that encodes how the dataset was built (for example, similarity
  to a list of known legitimate URLs) rather than something a live detector could rely on.
- If the CV metrics barely change without it, the model does not depend on it and dropping it costs little.
  If they drop clearly, the reported scores depend heavily on this one feature, and the report should say so.
- Caveat: the hyperparameters were tuned **with** the feature. If the group drops it, re-run
  section 6 on the reduced feature list before the final report.
- Nothing here uses the test set.

*Findings, run of 2026-11-17 (training split only):*
- `URLSimilarityIndex` has the highest single-feature ROC-AUC (0.9961). **Every legitimate training URL
  has exactly 100** (std 0), against 0.79 % of phishing URLs. It behaves like a label proxy for
  "legitimate", which suggests it was computed against the legitimate URL list used to build the dataset.
- Dropping it changes the tuned model's CV F1 from 0.99991 to 0.99965 and ROC-AUC from 1.00000 to 0.99998.
  The model barely depends on it, because **other features also nearly separate the classes on their own**
  (`LineOfCode` 0.9902, `NoOfExternalRef` 0.9880, `NoOfImage` 0.9795).
- So removing `URLSimilarityIndex` alone does not explain or remove the near-perfect scores. The
  page-content features (crawled HTML) deserve the same discussion. Group decision; the feature stays for now.

## 8. Final test-set report

Run once, after tuning and the group decisions are final (set `RUN_FINAL_TEST = True`).
Do not choose models, hyperparameters or features from this result.

In [ ]:
if RUN_FINAL_TEST:
    model = logreg.train(X_train, y_train)
    final = evaluate.evaluate(model, X_test, y_test)
    display(pd.Series(final).round(5))
else:
    print("Skipped: the test set stays untouched until final evaluation (RUN_FINAL_TEST = False).")

## Experiment log

Numbers are copied from the cell outputs above (rerun the notebook to regenerate them).

| Date | What changed | Why | CV or test | Key metrics (F1 / ROC-AUC / recall) | Commit |
|---|---|---|---|---|---|
| 2026-11-17 | Baseline: lbfgs, L2, C = 1, 50 original features | Starting point | CV (5-fold, full train split) | 0.99986 / 1.00000 / 0.99971 | not committed yet |
| 2026-11-17 | Grid search C x L1/L2 (liblinear); chose L2, C = 10 | Tune regularisation (4.9) | CV (5-fold, full train split) | 0.99991 / 1.00000 / 0.99983 | not committed yet |
| 2026-11-17 | Tuned model without `URLSimilarityIndex` (49 features) | Leakage evidence for the group | CV (5-fold, full train split) | 0.99965 / 0.99998 / 0.99947 | not committed yet |